Source paper: PI-Link: A Ground-Truth Dataset of Links Between Pull-Requests and Issues in GitHub

paper link: https://ieeexplore.ieee.org/abstract/document/10002372

dataset link: https://www.kaggle.com/datasets/zakareaalshara/android-closed-issues-20110101-20210101-clean

The dataset contains links between pull requests and issues collected from Android projects on GitHub. It is available on kaggle in the form of one JSON file. The code in "json_processing.py" transforms it into 3 dataframes: Pull_Requests, Issues and links between them.

In [2]:
import kagglehub

from json_processing import load_json_dataframes, construct_issue_links, normalise_labels

In [3]:
path = kagglehub.dataset_download("zakareaalshara/android-closed-issues-20110101-20210101-clean")

print("Path to dataset files:", path)

(issues, prs, links) = load_json_dataframes(path)
issue_links = construct_issue_links(issues, links)

Path to dataset files: C:\Users\dhali\.cache\kagglehub\datasets\zakareaalshara\android-closed-issues-20110101-20210101-clean\versions\3
Total JSON entries: 34732


Issues table:

In [4]:
print(f"Nr of issues: {len(issues)}")
print(f"Columns: {issues.columns.values.tolist()}")

Nr of issues: 34732
Columns: ['issue_no', 'issue_url', 'issue_title', 'issue_body', 'repository', 'created_at', 'closed_at', 'labels', 'comments']


In [5]:
first_row = issues.iloc[0]
print(f"First issue nr: {first_row['issue_no']}")
print(f"url: {first_row['issue_url']}")
print(f"title: {first_row['issue_title']}")

First issue nr: 14
url: https://github.com/05nelsonm/TorOnionProxyLibrary-Android/issues/14
title: App crashing sometimes when ServiceAction.RESTART is being executed.


In [6]:
print(f"body: {first_row['issue_body']}")

body: # Description
<!-- Short description of the issue  -->
App is crashing after Tor is restarted. It reaches 95% Bootstrapped, stalls, then crashes.

## Steps To Reproduce
<!--- Provide a link to a live example, or an unambiguous set of steps to -->
<!--- reproduce this bug. Include code to reproduce, if relevant          -->
Happens sporadically, and unsure if it's only something with sending the `ServiceAction` via the notification's `PendingIntent`, or if it's also happening with the `TorServiceController.restartTor` method, too.


In [7]:
print(f"repository: {first_row['repository']}")

repository: 05nelsonm/TorOnionProxyLibrary-Android


In [8]:
print(f"created at: {first_row['created_at']}")
print(f"closed at: {first_row['closed_at']}")

created at: 2020-06-26T17:24:15Z
closed at: 2020-07-02T21:27:15Z


In [9]:
print(f"labels: {first_row['labels']}")

labels: bug


In [10]:
print(f"comments:\n {first_row['comments']}")

comments:
 Stack trace:
```
2020-06-28 06:05:32.947 20706-21136/io.matthewnelson.sampleapp E/AndroidRuntime: FATAL EXCEPTION: Thread-5
    Process: io.matthewnelson.sampleapp, PID: 20706
    java.lang.StringIndexOutOfBoundsException: String index out of range: -1
        at java.lang.String.substring(String.java:1975)
        at net.freehaven.tor.control.TorControlConnection.handleEvent(TorControlConnection.java:228)
        at net.freehaven.tor.control.TorControlConnection.react(TorControlConnection.java:373)
        at net.freehaven.tor.control.TorControlConnection$ControlParseThread.run(TorControlConnection.java:343)
```

and

```
06-28 11:24:27.984 E/AndroidRuntime(13416): FATAL EXCEPTION: Thread-6
06-28 11:24:27.984 E/AndroidRuntime(13416): Process: io.matthewnelson.sampleapp, PID: 13416
06-28 11:24:27.984 E/AndroidRuntime(13416): java.util.ConcurrentModificationException
06-28 11:24:27.984 E/AndroidRuntime(13416): 	at java.util.ArrayList$Itr.next(ArrayList.java:860)
06-28 11:24:2

Linked_issues is a table assigning to each issue the issues that share at least one pr with it. It is the table storing the ground truth for finding connections between issues. It was constructed based on the data frames: prs and links.

In [11]:
print(f"Nr of entries: {len(issue_links)}")
print(f"Columns: {issue_links.columns.values.tolist()}")

Nr of entries: 8216
Columns: ['issue_no', 'related_issue_nos']


In [12]:
print(issue_links.head())

   issue_no              related_issue_nos
0        14        [8, 9, 13, 15, 20, 381]
1        33                  [15, 44, 294]
2        31               [18, 30, 37, 39]
3        52  [44, 51, 55, 56, 57, 59, 754]
4       101                      [98, 104]


Currently the issue is entered into issue_links only if it has documented link to some pr, which is why there are significantly less entries in this table comparing to Issues table.

## Label normalisation

In the raw dataset there are a lot of redundant labels, as can be seen below.

In [14]:
from collections import Counter

label_counts = Counter(
    label
    for labels in issues["labels"].dropna()
    for label in labels.split("|")
    if label
)

for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label}: {count}")

bug: 6583
enhancement: 2285
Bug: 1286
good first issue: 898
Android: 892
help wanted: 743
in progress: 592
[Type] Bug: 574
[QA]:Verified fixed: 526
android: 517
type: bug: 494
feature: 417
pr exists: 324
Resolution: Locked: 301
regression: 291
[Type] Enhancement: 287
framework: 281
Type: Bug: 270
[QA]:Normal issue: 259
approved: 252
state: verified fixed: 223
released: 220
[Type] Crash: 209
question: 199
type: bug/fix: 191
iOS: 190
triaged: 188
UI: 187
t/bug :bug:: 186
Status: Not started: 184
翻译完成: 180
🐞 bug: 176
Enhancement: 171
P1: 164
[Pri] High: 161
hacktoberfest: 160
tool: 150
wallet: 150
high priority: 148
release: 146
high-priority: 140
Core: 137
severe: crash: 132
Hacktoberfest: 130
f: material design: 129
os: android: 128
assigned: 126
welcome contribute: 126
Posting/Editing: 122
Type: Improvement: 121
[QA]:Major issue: 121
Reader: 120
documentation: 119
P2: 118
plugin: 117
p/Android: 112
bounty: 109
priority: p3: 109
eng:qa:verified: 109
t/enhancement ➕: 106
Priority: Essent

The raw `labels` field contains hundreds of free-text variants across repositories (e.g. `"type: bug"`, `"[Type] Bug"`, `"🐞 Bug"` all mean the same thing). To make labels usable for analysis and modelling, they are mapped to a small set of canonical group names organised into 6 dimensions:

| Dimension | Values | Example raw labels |
|---|---|---|
| **type** | `bug`, `crash`, `regression`, `enhancement`, `task`, `question`, `documentation` | `type: bug`, `[Type] Bug`, `feature request` |
| **priority** | `priority:critical`, `priority:high`, `priority:medium`, `priority:low` | `P1`, `HIGH PRIORITY`, `low-severity` |
| **status** | `status:in_progress`, `status:qa`, `status:resolved`, `status:stale` | `in progress`, `QA-verified`, `wontfix` |
| **platform** | `platform:android`, `platform:ios`, `platform:web`, `platform:linux` | `OS: Android`, `p/iOS 🍎` |
| **layer** | `layer:frontend`, `layer:backend`, `layer:infrastructure` | `UI`, `ux`, `backend`, `API`, `Build` |
| **community** | `community:good_first_issue`, `community:help_wanted`, `community:bounty` | `good first issue`, `Hacktoberfest` |

Each issue's `labels` field is replaced with a pipe-separated string of matched canonical names (e.g. `"bug|layer:frontend|priority:high"`). Labels that do not match any group are dropped. The normalisation is implemented in `json_processing.normalise_labels()`.

In [15]:
issues = normalise_labels(issues)
label_counts = Counter(
    label
    for labels in issues["labels"].dropna()
    for label in labels.split("|")
    if label
)

for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label}: {count}")

bug: 8601
enhancement: 3722
status:in_progress: 1531
platform:android: 1339
community:good_first_issue: 1206
status:qa: 1055
priority:high: 971
community:help_wanted: 743
layer:frontend: 731
priority:medium: 616
status:stale: 603
status:resolved: 534
crash: 430
priority:low: 408
platform:ios: 355
regression: 339
community:bounty: 309
question: 277
priority:critical: 238
documentation: 236
task: 196
layer:backend: 146
layer:infrastructure: 136
platform:linux: 36
platform:web: 32


In [16]:
issues.to_csv("issues.csv", index=False)
print("Saved issues.csv")

Saved issues.csv
